# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "auto-Nact-Ediv-vir"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f ]
out = [str(x) for x in [k for k in np.arange(0, 752)] if x not in run_list]
(' '.join((out)))

'6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 276 277 278 279 2

In [4]:
# # Load data from second infections
# file_list = [f for f in os.listdir(os.path.join(d, "raw"))  if runs in f and infection_type in f and comment in f]

# for f in tqdm(file_list):
#     filepath = os.path.join(os.path.join(d, "raw"), f)
#     with open(filepath, 'rb') as filename:  
#         import_dict = pickle.load(filename)

#     parameters_nets = []
#     prim_diff_bias_list = []
#     #sec_diff_bias_list = []
#     cell_series_list = []
#     # mean_prim_diff_bias = []
#     # std_prim_diff_bias = []
#     # mean_sec_diff_bias = []
#     # std_sec_diff_bias = []
#     # mean_cell_series = []
#     # std_cell_series = []
#     # mean_lineage_diff = []
#     # std_lineage_diff = []

#     parameters = np.array(import_dict["parameters"])
#     # prim_diff_bias = np.array(import_dict["prim_diff_bias"])
#     # sec_diff_bias = np.array(import_dict["sec_diff_bias"])
#     # cell_series = np.array(import_dict["cell_time_series"])
#     # lineage_diff = np.array(import_dict["lineage_diff"])
#     sim_sum = np.array(import_dict["summary_stats"])

#     # virs = np.unique(parameters[:,[4,7,13,14]], axis = 0)
    
#     # for i, vir in enumerate(virs): # Need to fix this to stop averaging over K_EI and K_EH
#     #     index = (parameters[:,4] == vir[0])*(parameters[:,7] == vir[1])*(parameters[:,13] == vir[2])*(parameters[:,14] == vir[3])
                
#     sim_sum_list.append(np.hstack((sim_sum, parameters)))

#         # mean_prim_diff_bias.append(np.mean(prim_diff_bias[index], axis = 0))
#         # std_prim_diff_bias.append(np.std(prim_diff_bias[index], axis = 0))

#         # mean_sec_diff_bias.append(np.mean(sec_diff_bias[index], axis = 0))
#         # std_sec_diff_bias.append(np.std(sec_diff_bias[index], axis = 0))
        
#         # mean_cell_series.append(np.mean(cell_series[index], axis = 0))
#         # std_cell_series.append(np.std(cell_series[index], axis = 0))
        
#         # mean_lineage_diff.append(np.mean(lineage_diff[index], axis = 0))
#         # std_lineage_diff.append(np.std(lineage_diff[index], axis = 0))

# # Save datasets
# ### (1) Summary stats
# np.save(os.path.join(d, "raw",comment+"_summary_stats"), np.vstack(sim_sum_list))
#     # np.save(os.path.join(d, "summary_stats","std",f[:-4]), std_sim_sum)
#     # ### (2) Differentiation bias
#     # np.save(os.path.join(d, "prim_diff_bias","mean",f[:-4]), mean_prim_diff_bias)
#     # np.save(os.path.join(d, "prim_diff_bias","std",f[:-4]), std_prim_diff_bias)
#     # np.save(os.path.join(d, "sec_diff_bias","mean",f[:-4]), mean_sec_diff_bias)
#     # np.save(os.path.join(d, "sec_diff_bias","std",f[:-4]), std_sec_diff_bias)
#     # ### (3) Cell time series
#     # np.save(os.path.join(d, "cell_time_series","mean",f[:-4]), mean_cell_series)
#     # np.save(os.path.join(d, "cell_time_series","std",f[:-4]), std_cell_series)
#     # ### (4) Lineage differentiation
#     # np.save(os.path.join(d, "lineage_diff","mean",f[:-4]), mean_lineage_diff)
#     # np.save(os.path.join(d, "lineage_diff","std",f[:-4]), std_lineage_diff)

In [8]:
num_cpu = 20
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names]).groupby(param_names_for_df[-len(NM_psis + EM_psis + act_psis + exp_psis):] + ['d_I', 'K_IE', 'b_I'], as_index=False).mean()

In [9]:
# Save datasets
### (1) Summary stats
mean_df.to_pickle(os.path.join(d, "raw", "stacked_data"+runs+"runs"+'-'+comment)+'.pkl')
del mean_df